<a href="https://colab.research.google.com/github/chandupradeep18/Building-with-the-Claude-API/blob/main/Claude_Text_Editing_Tool.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install anthropic openai

In [ ]:
from anthropic import Anthropic
from anthropic.types import ToolParam,Message
from openai import OpenAI
import requests,json,csv,os
from google.colab import userdata
from datetime import datetime,timedelta
nebius = OpenAI(base_url="https://api.tokenfactory.nebius.com/v1/",api_key=userdata.get("NEBIUS"))
nvidia = OpenAI(base_url="https://integrate.api.nvidia.com/v1",api_key=userdata.get('NVIDIA'))
claude = Anthropic(api_key=userdata.get('CLAUDE'))
ollama = OpenAI(base_url="https://ollama.com/v1",api_key=userdata.get('OLLAMA'))
nebius_model={'api':'nebius','model':"nvidia/Nemotron-3.5-Super-VL"}
nvidia_model={'api':'nvidia','model':"nvidia/nemotron-3.5-super-vl-120b-a12b"}
claude_model={'api':'claude','model':"claude-sonnet-4-5"}
ollama_gemma4_model={'api':'ollama','model':"gemma4:31b"}

In [ ]:
def add_user_message(messages, message):
    messages.append({"role": "user", "content": message.content if isinstance(message,Message) else message})
def add_assistant_message(messages, message):
    messages.append({"role": "assistant", "content": message.content if isinstance(message,Message) else message})
def collect_chat_history(response):
  file_exist=os.path.exists('/content/drive/MyDrive/Response/chat_history.csv')
  with open('/content/drive/MyDrive/Response/chat_history.csv','a',newline='') as save_response:
    writer=csv.DictWriter(save_response,fieldnames=response.keys())
    if not file_exist:
      writer.writeheader()
    writer.writerow(response)
def openai_client(client,messages,model,system=None,temperature=0.5,max_tokens=1800):
  payload = {"model": model['model'],"messages": messages,"temperature": temperature,"max_tokens": max_tokens,"stream": False,}
  if system:
    messages.append({"role": "system", "content": system})
  response = client.chat.completions.create(**payload)
  message = response.choices[0].message
  usage = response.usage
  details={ "Client": "OpenAi",'api':model['api'],'model':model['model'],"Time": datetime.now().isoformat(),"id": response.id,"role": message.role,"stop_reason": response.choices[0].finish_reason,
    "input_tokens": usage.prompt_tokens,"output_tokens": usage.completion_tokens,"total_tokens": usage.total_tokens,"user": messages[-1]["content"] ,"assistant": message.content}
  collect_chat_history(details)
  return response.choices[0].message.content
def claude_client(client,messages,model,system=None,max_tokens=1800,stop_sequences=[],tool=None):
  params = {"model": model['model'],"max_tokens": max_tokens,"messages": messages,"stop_sequences": stop_sequences}
  if tool:
    params["tools"] = tool
  if system:
    params["system"] = system
  response = client.messages.create(**params)
  details={'Client':'Claude','api':model['api'],'model':model['model'],'Time':datetime.now().isoformat(),'id':response.id,'role':response.role,'stop_reason':response.stop_reason,
           'input_tokens':response.usage.input_tokens,'output_tokens':response.usage.output_tokens,"total_tokens": response.usage.output_tokens+response.usage.input_tokens,
           'user':messages[-1]['content'],'assistant':response.content}
  collect_chat_history(details)
  return response

In [ ]:
# Implementation of the TextEditorTool
import os
import shutil
from typing import Optional, List
class TextEditorTool:
    def __init__(self, base_dir: str = "", backup_dir: str = ""):
        self.base_dir = base_dir or os.getcwd()
        self.backup_dir = backup_dir or os.path.join(self.base_dir, ".backups")
        os.makedirs(self.backup_dir, exist_ok=True)

    def _validate_path(self, file_path: str) -> str:
        abs_path = os.path.normpath(os.path.join(self.base_dir, file_path))
        if not abs_path.startswith(self.base_dir):
            raise ValueError(
                f"Access denied: Path '{file_path}' is outside the allowed directory"
            )
        return abs_path

    def _backup_file(self, file_path: str) -> str:
        if not os.path.exists(file_path):
            return ""
        file_name = os.path.basename(file_path)
        backup_path = os.path.join(
            self.backup_dir, f"{file_name}.{os.path.getmtime(file_path):.0f}"
        )
        shutil.copy2(file_path, backup_path)
        return backup_path

    def _restore_backup(self, file_path: str) -> str:
        file_name = os.path.basename(file_path)
        backups = [
            f for f in os.listdir(self.backup_dir) if f.startswith(file_name + ".")
        ]
        if not backups:
            raise FileNotFoundError(f"No backups found for {file_path}")

        latest_backup = sorted(backups, reverse=True)[0]
        backup_path = os.path.join(self.backup_dir, latest_backup)

        shutil.copy2(backup_path, file_path)
        return f"Successfully restored {file_path} from backup"

    def _count_matches(self, content: str, old_str: str) -> int:
        return content.count(old_str)

    def view(self, file_path: str, view_range: Optional[List[int]] = None) -> str:
        try:
            abs_path = self._validate_path(file_path)

            if os.path.isdir(abs_path):
                try:
                    return "\n".join(os.listdir(abs_path))
                except PermissionError:
                    raise PermissionError(
                        "Permission denied. Cannot list directory contents."
                    )

            if not os.path.exists(abs_path):
                raise FileNotFoundError("File not found")

            with open(abs_path, "r", encoding="utf-8") as f:
                content = f.read()

            if view_range:
                start, end = view_range
                lines = content.split("\n")

                if end == -1:
                    end = len(lines)

                selected_lines = lines[start - 1 : end]

                result = []
                for i, line in enumerate(selected_lines, start):
                    result.append(f"{i}: {line}")

                return "\n".join(result)
            else:
                lines = content.split("\n")
                result = []
                for i, line in enumerate(lines, 1):
                    result.append(f"{i}: {line}")

                return "\n".join(result)

        except UnicodeDecodeError:
            raise UnicodeDecodeError(
                "utf-8",
                b"",
                0,
                1,
                "File contains non-text content and cannot be displayed.",
            )
        except ValueError as e:
            raise ValueError(str(e))
        except PermissionError:
            raise PermissionError("Permission denied. Cannot access file.")
        except Exception as e:
            raise type(e)(str(e))

    def str_replace(self, file_path: str, old_str: str, new_str: str) -> str:
        try:
            abs_path = self._validate_path(file_path)

            if not os.path.exists(abs_path):
                raise FileNotFoundError("File not found")

            with open(abs_path, "r", encoding="utf-8") as f:
                content = f.read()

            match_count = self._count_matches(content, old_str)

            if match_count == 0:
                raise ValueError(
                    "No match found for replacement. Please check your text and try again."
                )
            elif match_count > 1:
                raise ValueError(
                    f"Found {match_count} matches for replacement text. Please provide more context to make a unique match."
                )

            # Create backup before modifying
            self._backup_file(abs_path)

            # Perform the replacement
            new_content = content.replace(old_str, new_str)

            with open(abs_path, "w", encoding="utf-8") as f:
                f.write(new_content)

            return "Successfully replaced text at exactly one location."

        except ValueError as e:
            raise ValueError(str(e))
        except PermissionError:
            raise PermissionError("Permission denied. Cannot modify file.")
        except Exception as e:
            raise type(e)(str(e))

    def create(self, file_path: str, file_text: str) -> str:
        try:
            abs_path = self._validate_path(file_path)

            # Check if file already exists
            if os.path.exists(abs_path):
                raise FileExistsError(
                    "File already exists. Use str_replace to modify it."
                )

            # Create parent directories if they don't exist
            os.makedirs(os.path.dirname(abs_path), exist_ok=True)

            # Create the file
            with open(abs_path, "w", encoding="utf-8") as f:
                f.write(file_text)

            return f"Successfully created {file_path}"

        except ValueError as e:
            raise ValueError(str(e))
        except PermissionError:
            raise PermissionError("Permission denied. Cannot create file.")
        except Exception as e:
            raise type(e)(str(e))

    def insert(self, file_path: str, insert_line: int, new_str: str) -> str:
        try:
            abs_path = self._validate_path(file_path)

            if not os.path.exists(abs_path):
                raise FileNotFoundError("File not found")

            # Create backup before modifying
            self._backup_file(abs_path)

            with open(abs_path, "r", encoding="utf-8") as f:
                lines = f.readlines()

            # Handle line endings
            if lines and not lines[-1].endswith("\n"):
                new_str = "\n" + new_str

            # Insert at the beginning if insert_line is 0
            if insert_line == 0:
                lines.insert(0, new_str + "\n")
            # Insert after the specified line
            elif insert_line > 0 and insert_line <= len(lines):
                lines.insert(insert_line, new_str + "\n")
            else:
                raise IndexError(
                    f"Line number {insert_line} is out of range. File has {len(lines)} lines."
                )

            with open(abs_path, "w", encoding="utf-8") as f:
                f.writelines(lines)

            return f"Successfully inserted text after line {insert_line}"

        except ValueError as e:
            raise ValueError(str(e))
        except PermissionError:
            raise PermissionError("Permission denied. Cannot modify file.")
        except Exception as e:
            raise type(e)(str(e))

    def undo_edit(self, file_path: str) -> str:
        try:
            abs_path = self._validate_path(file_path)

            if not os.path.exists(abs_path):
                raise FileNotFoundError("File not found")

            return self._restore_backup(abs_path)

        except ValueError as e:
            raise ValueError(str(e))
        except FileNotFoundError:
            raise FileNotFoundError("No previous edits to undo")
        except PermissionError:
            raise PermissionError("Permission denied. Cannot restore file.")
        except Exception as e:
            raise type(e)(str(e))
# Make the text edit schema based on the model version being used
def get_text_edit_schema():
    return {
        "type": "text_editor_20250728",
        "name": "str_replace_based_edit_tool",
    }

In [ ]:
# Process Tool Call Requests
import json

text_editor_tool = TextEditorTool()


def run_tool(tool_name, tool_input):
    if tool_name == "str_replace_editor":
        command = tool_input["command"]
        if command == "view":
            return text_editor_tool.view(
                tool_input["path"], tool_input.get("view_range")
            )
        elif command == "str_replace":
            return text_editor_tool.str_replace(
                tool_input["path"], tool_input["old_str"], tool_input["new_str"]
            )
        elif command == "create":
            return text_editor_tool.create(tool_input["path"], tool_input["file_text"])
        elif command == "insert":
            return text_editor_tool.insert(
                tool_input["path"],
                tool_input["insert_line"],
                tool_input["new_str"],
            )
        elif command == "undo_edit":
            return text_editor_tool.undo_edit(tool_input["path"])
        else:
            raise Exception(f"Unknown text editor command: {command}")
    else:
        raise Exception(f"Unknown tool name: {tool_name}")


def run_tools(message):
    tool_requests = [block for block in message.content if block.type == "tool_use"]
    tool_result_blocks = []

    for tool_request in tool_requests:
        try:
            tool_output = run_tool(tool_request.name, tool_request.input)
            tool_result_block = {
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": json.dumps(tool_output),
                "is_error": False,
            }
        except Exception as e:
            tool_result_block = {
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": f"Error: {e}",
                "is_error": True,
            }

        tool_result_blocks.append(tool_result_block)

    return tool_result_blocks

def conversation(messages,tools=[], fine_grained=False):
  while True:
    if fine_grained:
       with claude.beta.messages.stream(model=claude_model['model'],messages=messages,max_tokens=1000,
       tools=tools,betas=['fine-grained-tool-streaming-2025-05-14'] if fine_grained else [],
       ) as streams:
         for chunk in streams:
          if chunk.type == "text":
            print(chunk.text, end="")
          if chunk.type == "content_block_start":
            if chunk.content_block.type == "tool_use":
                print(f'\n>>> Tool Call: "{chunk.content_block.name}"')
          if chunk.type == "input_json" and chunk.partial_json:
                print(chunk.partial_json, end="")
         response = streams.get_final_message()
    else:
      response=claude_client(claude,messages,claude_model,max_tokens=1000,tool=tools)
    add_assistant_message(messages,response.content)
    if response.stop_reason!='tool_use':
      break
    tool_results=run_tools(response)
    add_user_message(messages, tool_results)
  return messages

In [ ]:
web_search_schema = {
    "type": "web_search_20260318",
    "name": "web_search",
    "max_uses": 5,
    #"allowed_domains": ["nih.gov"],
}

In [ ]:
chat_history=[]
add_user_message(chat_history,'Tell me about Django Rest Framework')
response=conversation(chat_history,tools=[web_search_schema],fine_grained=True)

/tmp/ipykernel_1003/2428987228.py:3: DeprecationWarning: The model 'claude-sonnet-4-5' is deprecated and will reach end-of-life on November 30th, 2026.
Please migrate to a newer model. Visit https://docs.anthropic.com/en/docs/resources/model-deprecations for more information.
  response=conversation(chat_history,tools=[web_search_schema],fine_grained=True)


Django Rest Framework (DRF) is a powerful and flexible toolkit for building Web APIs in Django. Here's a comprehensive overview:

## What is Django Rest Framework?

Django Rest Framework is an open-source Python library that sits on top of Django and provides a set of tools for building RESTful APIs quickly and efficiently. It's one of the most popular solutions for API development in the Django ecosystem.

## Key Features

**1. Serialization**
- Converts complex data types (like Django models) to Python data types that can be easily rendered into JSON, XML, or other content types
- Handles deserialization for parsing incoming data and validating it

**2. Authentication & Permissions**
- Multiple authentication schemes (Token, Session, OAuth, JWT)
- Flexible permission classes to control access
- Built-in support for throttling to prevent abuse

**3. Browsable API**
- Automatically generated, web-browsable API interface
- Makes testing and debugging much easier during development

**4.

In [ ]:
response[-1]['content']

[ParsedBetaTextBlock(citations=None, text="Django Rest Framework (DRF) is a powerful and flexible toolkit for building Web APIs in Django. Here's a comprehensive overview:\n\n## What is Django Rest Framework?\n\nDjango Rest Framework is an open-source Python library that sits on top of Django and provides a set of tools for building RESTful APIs quickly and efficiently. It's one of the most popular solutions for API development in the Django ecosystem.\n\n## Key Features\n\n**1. Serialization**\n- Converts complex data types (like Django models) to Python data types that can be easily rendered into JSON, XML, or other content types\n- Handles deserialization for parsing incoming data and validating it\n\n**2. Authentication & Permissions**\n- Multiple authentication schemes (Token, Session, OAuth, JWT)\n- Flexible permission classes to control access\n- Built-in support for throttling to prevent abuse\n\n**3. Browsable API**\n- Automatically generated, web-browsable API interface\n- Ma